# Instalando pacotes

In [18]:
from io import BytesIO
import sys
!{sys.executable} -m pip install fsspec s3fs oci ocifs 
!{sys.executable} -m pip install pandas numpy==1.24.3
!{sys.executable} -m pip uninstall pyarrow -y
!{sys.executable} -m pip install pyarrow==12.0.1

# Carregando pacotes

In [1]:
import oci
import ocifs
import pandas as pd
import sys
import os
import pyarrow
# Funcoes customizadas
import configs.function_basic as funcoes

# Conexão ao repositório via OCI

In [2]:
#Buckets e nomes de saída nuvem = "oci://"
namespace = "@grxzqsiaote6/"
pasta_in = 'Feature_store/book_variaveis_02.parquet'
pasta_in_trusted = 'base_telco/'
pasta_out = 'Feature_store/'

bucket_book = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_in}" 
bucket_trusted = f"oci://TRUSTED{namespace}{pasta_in_trusted}"
bucket_feature_store = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_out}"

# Carregando databases

## Book_02

In [3]:
from ocifs import OCIFileSystem

fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_02.parquet")
files = fs.ls(bucket_book)

print(files)

['BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_02.parquet']


In [7]:
# Carregando book_02
df_book_02 = pd.read_parquet(
    bucket_book, #"oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_02.parquet",
    storage_options={"config": "~/.oci/config"})
print("Book 02 data shape:", df_book_02.shape)

Book 02 data shape: (3734429, 35)


In [9]:
df_book_02.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3734429 entries, 0 to 3734428
Data columns (total 35 columns):
 #   Column              Dtype         
---  ------              -----         
 0   Ano                 Int32         
 1   Mes                 Int8          
 2   FLAG_INSTALACAO     boolean       
 3   ProductMigration    string        
 4   SCORE_01            float32       
 5   SCORE_02            float32       
 6   FPD                 Int32         
 7   NUM_CPF             object        
 8   SAFRA               Int32         
 9   SCORE_RATE          float32       
 10  SCORE_AVG           float32       
 11  SCORE_DIFF          float32       
 12  SCORE_MIN           float32       
 13  SAFRA_ANO           int32         
 14  SAFRA_MES           int32         
 15  DATA_DE_NASCIMENTO  datetime64[ns]
 16  var_03              float64       
 17  var_04              float64       
 18  var_05              float64       
 19  var_09              float64       
 20  Ti

## Base Dados Telco

In [10]:
fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://TRUSTED@grxzqsiaote6/base_dados_cadastrais/")
files = fs.ls(bucket_trusted)

print(files)

['TRUSTED@grxzqsiaote6/base_telco/_SUCCESS', 'TRUSTED@grxzqsiaote6/base_telco/ts_proc_partition=20260312023206']


In [11]:
## Carregando todos arquivos em parquet de uma pasta
df_dados_telco = pd.read_parquet(
    bucket_trusted, #"oci://TRUSTED@grxzqsiaote6/base_dados_cadastrais/",
    storage_options={"config": "~/.oci/config"})

print('Base Dados Telco data shape:', df_dados_telco.shape)

Base Dados Telco data shape: (1367104, 78)


In [12]:
df_dados_telco.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1367104 entries, 0 to 1367103
Data columns (total 78 columns):
 #   Column              Non-Null Count    Dtype   
---  ------              --------------    -----   
 0   ts_proc             1367104 non-null  object  
 1   NUM_CPF             1367104 non-null  object  
 2   SAFRA_ANO           1367104 non-null  int32   
 3   SAFRA_MES           1367104 non-null  int32   
 4   IsSetup             1367104 non-null  bool    
 5   IsFPD               1321168 non-null  object  
 6   ProductDescription  1367104 non-null  object  
 7   ProductMigration    1308974 non-null  object  
 8   Var26               1365809 non-null  float32 
 9   Var27               1365809 non-null  float32 
 10  Var28               1365809 non-null  float32 
 11  Var29               1365809 non-null  float32 
 12  Var30               1365809 non-null  float32 
 13  Var31               1365809 non-null  float32 
 14  Var32               1365809 non-null  float32 
 15

Como na analise exploratoria vimos que estes são todos valores numericos continuos, iremos transformar todas as `var_x` em numericas.

Iremos ajustar aqui pois o book_02 possui este mesmo padrao de colunas mas nao sao numericas.

In [13]:
funcoes.convert_var_columns_to_numeric(df_dados_telco, inplace=True)

## Merge dos Datasets

In [14]:
# Primeiro vamos transformar a coluna SAFRA para o mesmo formato de book_02
df_dados_telco['SAFRA'] = df_dados_telco['SAFRA'].astype('int64')

In [15]:
cols_to_drop = [
    col for col in df_dados_telco.columns
    if col in df_book_02.columns
    and col not in ['SAFRA', 'NUM_CPF']
]

df_dados_telco_clean = df_dados_telco.drop(columns=cols_to_drop)

df_book_03 = pd.merge(
    df_book_02,
    df_dados_telco_clean,
    how='left',
    on=['SAFRA', 'NUM_CPF']
)

In [16]:
# Sanity check
df_book_02.shape[0] == df_book_03.shape[0]

True

In [17]:
funcoes.generate_metadata(df_book_03)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,IsFPD,object,2435276,65.21,2
1,Var41,float32,2398626,64.23,10008
2,Var31,float32,2398626,64.23,9775
3,Var37,float32,2398626,64.23,54
4,Var36,float32,2398626,64.23,96
...,...,...,...,...,...
103,DISPENSADO,Int64,0,0.00,2
104,REGIAO_POSTAL,object,0,0.00,11
105,SUB_REGIAO_POSTAL,object,0,0.00,101
106,REGIAO_POSTAL_TXT,object,0,0.00,11


## Feature Engineer


### Bloco 01

Deletaremos as colunas que possuem cardinalidade igual a 1

Como este dataset possui muitas colunas e o trabalho de entender uma por uma é exaustivo, primeiro iremos verificar se temos colunas com possuem exatamente o mesmo valor para cada linha

#### Bloco 01

In [18]:
funcoes.drop_single_cardinality_columns(df_book_03)

🧹 Colunas removidas (cardinalidade = 1): 0


,Ano,Mes,FLAG_INSTALACAO,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,SAFRA,SCORE_RATE,...,Var85,Var86,Var87,Var88,Var89,Var90,Var91,Var92,Var93,ts_proc_partition
0,2024,10,True,Aquisição,2.0,1.0,1,ZZZZZZZX7T9,202410,0.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,10,False,<NA>,562.0,559.0,<NA>,ZZZZZZZ8TZ8,202410,0.994662,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,10,False,<NA>,585.0,559.0,<NA>,ZZZZZZW9XWN,202410,0.955556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,10,True,PRE,562.0,636.0,0,ZZZZZX7XWY8,202410,1.131673,...,304.0,2.0,3.0,1.0,50.0,0.86,2.0,1.0,1.0,20260312023206
4,2024,10,True,Aquisição,538.0,570.0,1,ZZZZZX8TTUZ,202410,1.059480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3734424,2025,3,True,PRE,616.0,630.0,0,9999888YYU9,202503,1.022727,...,304.0,4.0,3.0,6.0,0.0,1.56,6.0,4.0,0.0,20260312023206
3734425,2025,3,True,PRE,627.0,649.0,0,9999889ZN9X,202503,1.035088,...,304.0,3.0,2.0,1.0,0.0,1.16,5.0,1.0,0.0,20260312023206
3734426,2025,3,True,Aquisição,561.0,661.0,0,999997YY7N8,202503,1.178253,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3734427,2025,3,True,PRE,577.0,611.0,1,999998YZYNW,202503,1.058926,...,1.0,304.0,304.0,304.0,304.0,304.00,304.0,304.0,304.0,20260312023206


In [19]:
# Conferindo colunas que possuem o mesmo valor
colunas_duplicadas = funcoes.find_duplicate_columns(df_book_03)
print("Colunas com valores duplicados:", colunas_duplicadas)

Colunas com valores duplicados: []


#### Ajustando os tipos de dados

Durante o processo inteiro os tipos de dados foram criadas da forma correta.

Não iremos realizar mais nenhum tratamento destes dados, pois são todos numericos continuos. A partir daqui iremos revisar apenas com a solicitação do time de Ciencia de Dados, caso alguma variável se mostre interessante.

In [20]:
# Criando novo dataset
book_variaveis_03 = df_book_03.copy()

In [21]:
# Salvando o dataframe em parquet
book_variaveis_03.to_parquet(
    f"{bucket_feature_store}book_variaveis_03.parquet",
    engine="pyarrow",
    compression="snappy",
#    partition_cols=["SAFRA"],
    index=False,
    storage_options={"config": "~/.oci/config"}
)